# 24B The MCP tool server
The catalog is closed, every tool has an action class, and scope is checked by the server on every call.

In [1]:
import sys, json, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "creditwatch").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, yaml
from creditwatch import (approvals, audit, catalog, channel, clock, econ, evals, heartbeat, llm, loop,
                         memory, panel, policy, signals, skills, standin, workspace)
P = yaml.safe_load(open(ROOT / "params.yaml"))
FX = ROOT / P["data"]["fixture_dir"]
USERS = json.load(open(FX / "users.json"))
MANDATES = json.load(open(ROOT / P["data"]["mandates"]))
REG = json.load(open(ROOT / P["data"]["registry"]))
PANEL = panel.load_panel(ROOT / P["data"]["panel"])
M = P["heartbeat"]["materiality"]
NOW = clock.clock()                       # LANTERN_CLOCK, never the system clock
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20); pd.set_option("display.max_colwidth", 60)
print("fixture mode:", P["data"]["use_fixture"], "| clock:", NOW.isoformat())

fixture mode: True | clock: 2026-10-15T06:00:00-04:00


In [2]:
cat = pd.DataFrame([(t, c, k or "(session)", catalog.policy_for(t)) for t, (c, k) in catalog.CATALOG.items()],
                   columns=["tool", "action class", "scope", "policy"])
print(cat.to_string(index=False))

                tool   action class     scope    policy
       list_mandates           READ (session)      auto
        bdc_holdings           READ    public      auto
         bdc_signals           READ    public      auto
          get_filing           READ (session)      auto
     mandate_profile           READ   mandate      auto
      watchlist_read           READ   mandate      auto
         memory_read           READ   mandate      auto
memory_propose_write INTERNAL_WRITE   mandate    notify
     watchlist_write INTERNAL_WRITE   mandate   approve
         draft_email          DRAFT   mandate      auto
    request_approval        CONTROL (session) principal
          send_email  EXTERNAL_SEND   mandate   approve


In [3]:
# Your server must declare exactly this catalog. A conformance check on a server with three mistakes:
mine = {t: c for t, (c, _) in catalog.CATALOG.items()}
mine.pop("watchlist_read"); mine["export_all_mandates"] = "READ"; mine["send_email"] = "INTERNAL_WRITE"
for p in catalog.conformance(mine):
    print("  ", p)

   missing tool: watchlist_read
   extra tool (the catalog is closed): export_all_mandates
   send_email: class INTERNAL_WRITE, expected EXTERNAL_SEND


In [4]:
s = channel.login("priya", "priya-fixture-token", "mandate_a", USERS)
led = audit.Ledger(clock=clock.clock)
tools = standin.read_tools(s, led, PANEL, MANDATES, M)        # four READ tools over the fixture
print("public signals:", len(tools["bdc_signals"](scope="public")))
print("own mandate:", tools["mandate_profile"](scope="mandate_a")["client"])
for call in [lambda: tools["mandate_profile"](scope="mandate_b"), lambda: tools["mandate_profile"](scope="public"),
             lambda: channel.require_host("https://example.com/exfil?x=1", led)]:
    try:
        call()
    except (channel.ScopeError, PermissionError) as e:
        print("refused:", type(e).__name__, e)
print(pd.DataFrame(led.find("policy_block"))[["event_id", "kind", "rule_id", "scope", "host"]].fillna(""))

public signals: 6
own mandate: Harbor State Employees' Retirement System
refused: ScopeError scope 'mandate_b' not allowed
refused: ScopeError scope 'public' not allowed
refused: PermissionError host 'example.com' not allowed
   event_id          kind  rule_id      scope         host
0  ev-00003  policy_block  R-SCOPE  mandate_b             
1  ev-00004  policy_block  R-SCOPE     public             
2  ev-00005  policy_block   R-HOST             example.com


## Wrapping it in MCP (SDK v2)
```python
from mcp.server import MCPServer            # pip install "mcp[cli]>=2.1,<3"
mcp = MCPServer("creditwatch-tools")

@mcp.tool()
def bdc_signals(scope: str) -> list[dict]:
    """Within-filing credit signals with evidence (READ)."""
    return tools["bdc_signals"](scope=scope)  # scope checked inside, every call
```
Type hints become the input schema and the docstring becomes the description. Run it with `mcp dev services/mcp/server.py` to open the Inspector. The cell below runs only if `mcp` is installed.

In [5]:
import importlib.util, asyncio
if importlib.util.find_spec("mcp") is None:
    print("mcp not installed: skipped (pip install 'mcp[cli]>=2.1,<3')")
else:
    try:
        from mcp.server import MCPServer
        from mcp import Client
        srv = MCPServer("creditwatch-demo")
        @srv.tool()
        def bdc_signals(scope: str) -> list[dict]:
            """Within-filing credit signals with evidence (READ)."""
            return tools["bdc_signals"](scope=scope)
        async def demo():
            async with Client(srv) as c:                  # in memory: no transport
                return await c.call_tool("bdc_signals", {"scope": "public"})
        print(await demo())  # Jupyter already runs an event loop
    except Exception as e:
        print("MCP demo failed:", type(e).__name__, e, "(check the SDK version against the v2 docs)")

meta={'io.modelcontextprotocol/serverInfo': {'name': 'creditwatch-demo', 'version': ''}} content=[TextContent(type='text', text='{\n  "adsh": "0009000012-26-000121",\n  "name": "FIXTURE HOTEL SENIOR LENDING",\n  "rule": "EXTENDED",\n  "identifier": "Pinecrest Learning Group Inc., First Lien Term Loan",\n  "row": "105",\n  "prior_row": "505",\n  "before": "2029-09-30",\n  "after": "2031-09-30"\n}', annotations=None, meta=None), TextContent(type='text', text='{\n  "adsh": "0009000012-26-000121",\n  "name": "FIXTURE HOTEL SENIOR LENDING",\n  "rule": "MARKDOWN",\n  "identifier": "Brightwater Fitness Holdings LLC, First Lien Term Loan",\n  "row": "101",\n  "prior_row": "501",\n  "before": 99.0,\n  "after": 93.5\n}', annotations=None, meta=None), TextContent(type='text', text='{\n  "adsh": "0009000012-26-000121",\n  "name": "FIXTURE HOTEL SENIOR LENDING",\n  "rule": "PIK_UP",\n  "identifier": "Oakmont Home Services LLC, First Lien Term Loan",\n  "row": "104",\n  "prior_row": "504",\n  "befor

**Your turn.** Build `services/mcp/` with exactly the Appendix C catalog, Pydantic schemas, typed errors and a contract test per tool (Part 1). `catalog.conformance` must return `[]` for your server's registry.

(.venv311) sailokeshreddynandavarapu@Sais-MacBook-Pro-2 DAMG-7245-Project-Lantern-Case-Study-Lab3B-Team-2 % cd cs3b
source .venv311/bin/activate

PYTHONPATH=src python -c "from services.mcp.server import REGISTRY; from creditwatch.catalog import conformance; print(conformance(REGISTRY))"
[]

In [8]:
# ROOT, USERS, MANDATES, channel, audit, and clock
# were loaded by the setup cell.
sys.path.insert(0, str(ROOT))

from mcp import Client
from services.mcp.server import create_server

session = channel.login(
    "priya",
    "priya-fixture-token",
    "mandate_a",
    USERS,
)

server = create_server(
    session=session,
    ledger=audit.Ledger(clock=clock.clock),
    mandates=MANDATES,
    root=ROOT,
)

async with Client(server) as client:
    result = await client.call_tool(
        "mandate_profile",
        {"scope": "mandate_a"},
    )

print(result.structured_content)

{'ok': True, 'data': {'client': "Harbor State Employees' Retirement System", 'positions': {'FIXTURE GOLF CAPITAL CORP': 42.0, 'FIXTURE HOTEL SENIOR LENDING': 18.0, 'FIXTURE JULIETT LENDING BDC': 35.5}}}
